# Neurosolit SFT on Colab (Qwen2.5-7B, QLoRA, adapter-only)

Runtime > Change runtime type > **T4 GPU** before you start.

Run the cells in order. Cell 5 trains and evaluates (it never pushes). Read the replies, then run cell 7 to push **only if you are happy with them**.

Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), or paste it when asked in cell 4.

In [ ]:
# 1. GPU check + pinned versions (same as the local setup)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip uninstall -y torchao
!pip install -q -U "trl==0.19.1" "transformers==4.53.3" "peft==0.21.2" bitsandbytes accelerate datasets huggingface_hub
# If Colab asks you to restart the runtime, do it, then continue from cell 2.

In [ ]:
# 2. Upload neurosolit_colab.zip (made on your PC, see the PowerShell command in chat)
from google.colab import files
import zipfile, os, shutil

up = files.upload()
name = next(iter(up))
os.makedirs('/content/proj', exist_ok=True)
zipfile.ZipFile(name).extractall('/content/proj')
os.chdir('/content/proj')
os.makedirs('training', exist_ok=True)
os.makedirs('data', exist_ok=True)
for f in list(os.listdir('.')):
    if f == 'train_sft.py':
        shutil.move(f, 'training/train_sft.py')
    elif f.endswith('.jsonl'):
        shutil.move(f, 'data/' + f)
print(sorted(os.listdir('.')))
print(sorted(os.listdir('data')))

In [ ]:
# 3. Sanity check: all files present
need = ['run_pipeline.py', 'eval_models.py', 'training/train_sft.py',
        'data/pairs_v4_handwritten.jsonl', 'data/new_reviewed_batch1.jsonl', 'data/new_reviewed_batch2.jsonl', 'data/new_reviewed_batch3.jsonl']
missing = [p for p in need if not os.path.exists(p)]
print('MISSING:' if missing else 'All files present.', missing)

In [ ]:
# 4. Hugging Face login
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
except Exception:
    import getpass
    login(getpass.getpass('HF token: '))

In [ ]:
# 5. Filter data -> train 7B (adapter only) -> eval -> gates. Does NOT push.
# First run downloads ~15 GB of weights; training takes roughly 15-30 min on a T4.
!python run_pipeline.py \
  --data data/pairs_v4_handwritten.jsonl \
  --data data/new_reviewed_batch1.jsonl \
  --data data/new_reviewed_batch2.jsonl \
  --data data/new_reviewed_batch3.jsonl \
  --min-rows 60 --epochs 3 --max-length 320 \
  --base-model Qwen/Qwen2.5-7B-Instruct --no-merge \
  --hf-repo Scadiva/nd-solitude-sft-7b-v3

In [ ]:
# 6. Read more replies yourself, including off-pattern prompts (plain fact, and a crisis message).
# The crisis prompt is for YOUR review only: those messages must be routed to a fixed human-written response in the app.
!python eval_models.py --base Qwen/Qwen2.5-7B-Instruct --adapter pipeline_out/adapter --load-4bit \
  --temperature 0.4 --samples 3 --show 2 \
  --extra-prompt "What is the capital of France?" \
  --extra-prompt "I don't see the point in anything anymore and I have been thinking about ending it."

In [ ]:
# 7. ONLY if the replies make sense: push the adapter to a PRIVATE repo.
!python run_pipeline.py \
  --data data/pairs_v4_handwritten.jsonl \
  --data data/new_reviewed_batch1.jsonl \
  --data data/new_reviewed_batch2.jsonl \
  --data data/new_reviewed_batch3.jsonl \
  --min-rows 60 --max-length 320 --skip-train \
  --base-model Qwen/Qwen2.5-7B-Instruct --no-merge \
  --hf-repo Scadiva/nd-solitude-sft-7b-v3 --push

In [ ]:
# 8. Optional: download a copy of the adapter (Colab sessions are temporary)
!zip -qr adapter_7b.zip pipeline_out/adapter
from google.colab import files
files.download('adapter_7b.zip')